# Notebook 1 — Data preparation and classical baselines
**Paper:** Cross-corpus Roman Urdu sentiment classification (Jawairia Iftikhar)

What this notebook does, in order:
1. Loads the four dataset files you upload (CSV or Excel) and recognises each one from its columns.
2. Cleans each corpus, removes cross-corpus duplicates (UCI → Aimlab → RUWV-NSR → Khubaib01), masks phone numbers and emails.
3. Saves the shared sentences for the label-agreement analysis (RQ3).
4. Makes fixed 80/10/10 splits and saves them — **every later notebook reuses these files**.
5. Trains two classical baselines (word TF-IDF and character TF-IDF, logistic regression) on each source corpus, tests on every corpus, and saves all predictions.
6. Builds the generalisation-drop table with 95% bootstrap confidence intervals.

**Before running:** click the folder icon on the left and upload your 4 dataset files (UCI, Aimlab, RUWV-NSR reviews, Khubaib01).
File names do not matter. Then Runtime → Run all. No GPU needed. About 15–30 minutes.
Outputs are saved inside this Colab session; the last cell downloads them as a zip — **keep that zip**.

## 1. Setup

In [ ]:
import sys, io, os, re, glob, json, zipfile, unicodedata, itertools, time
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

## 2. Settings

In [ ]:
DATA_DIR = '/content'            # where you uploaded the 4 files (Colab's file panel)
OUT = '/content/roman_urdu_paper'
SPLIT_SEED = 42                 # fixed once, never change after the first run
SEEDS = [13, 42, 2026]          # training seeds (affect the size-matched sample)
NEAR_DUP_COS = 0.9              # near-duplicate threshold (char 3-5-gram TF-IDF cosine)
NEAR_DUP_MIN_WORDS = 4          # only sentences with >= 4 words are checked for near-duplicates
BOOTSTRAP = 1000
ORDER = ['uci', 'aimlab', 'ruwv', 'khubaib']   # priority order: earlier corpora keep shared sentences
# Final sizes from the audit (9-19 Oct 2026; one empty UCI cell is dropped). If yours differ, a source file has changed — tell me before continuing.
EXPECTED_FINAL = {'uci': 19376, 'aimlab': 11353, 'ruwv': 20401, 'khubaib': 70581}

TRACKS = {   # name: (corpora, classes)
    '3class': (['uci', 'ruwv', 'khubaib'], ['neg', 'neu', 'pos']),
    'binary': (['uci', 'aimlab', 'ruwv', 'khubaib'], ['neg', 'pos']),
}
for sub in ['splits', 'overlaps', 'predictions', 'results']:
    os.makedirs(f'{OUT}/{sub}', exist_ok=True)

## 3. Find and load your four files
Each file is recognised by its columns:
- Khubaib01: `message`, `label`  ·  Aimlab: `Sentences`, `Labels`  ·  RUWV-NSR: `ROMAN URDU REVIEWS`, `SENTIMENT`
- UCI: no header row; second column holds Positive / Negative / Neutral

If Khubaib01 or Aimlab is not uploaded, it is loaded from Hugging Face instead.

In [ ]:
def read_any(path, header='infer'):
    if path.lower().endswith(('.xlsx', '.xls')):
        return pd.read_excel(path, header=0 if header == 'infer' else header)
    raw = open(path, 'rb').read()
    for enc in ['utf-8', 'utf-8-sig', 'latin-1']:   # RUWV-NSR needs latin-1
        try:
            return pd.read_csv(io.StringIO(raw.decode(enc)), header=header)
        except UnicodeDecodeError:
            continue
    raise ValueError(f'Could not read {path}')

LABEL_WORDS = {'positive', 'negative', 'neutral', 'neative'}
raw = {}
for path in sorted(glob.glob(f'{DATA_DIR}/*.csv') + glob.glob(f'{DATA_DIR}/*.xlsx')):
    df = read_any(path)
    df.columns = [c.strip() if isinstance(c, str) else c for c in df.columns]
    if {'message', 'label'} <= set(df.columns):
        name, t, l = 'khubaib', 'message', 'label'
    elif {'Sentences', 'Labels'} <= set(df.columns):
        name, t, l = 'aimlab', 'Sentences', 'Labels'
    elif {'ROMAN URDU REVIEWS', 'SENTIMENT'} <= set(df.columns):
        name, t, l = 'ruwv', 'ROMAN URDU REVIEWS', 'SENTIMENT'
    else:
        df = read_any(path, header=None)
        if df.shape[1] >= 2 and df[1].astype(str).str.strip().str.lower().isin(LABEL_WORDS).mean() > 0.9:
            name, t, l = 'uci', 0, 1
        else:
            print(f'skipped (not one of the 4 datasets): {os.path.basename(path)}')
            continue
    out = df[[t, l]].copy()
    out.columns = ['text', 'raw_label']
    raw[name] = out
    print(f'{name:8s} <- {os.path.basename(path)}  ({len(out):,} rows)')

for name, repo, t, l in [('aimlab', 'Aimlab/Sentiment-Analysis-Roman-Urdu', 'Sentences', 'Labels'),
                         ('khubaib', 'Khubaib01/RomanUrdu-NLP-Sentiment-Corpus', 'message', 'label')]:
    if name not in raw:
        from datasets import load_dataset
        df = load_dataset(repo, split='train').to_pandas()
        raw[name] = df[[t, l]].rename(columns={t: 'text', l: 'raw_label'})
        print(f'{name:8s} <- Hugging Face {repo}  ({len(raw[name]):,} rows)')

missing = [k for k in ORDER if k not in raw]
assert not missing, f'Missing dataset(s): {missing}. Upload the file(s) to the file panel and run this cell again.'
EXPECTED_RAW = {'uci': 20229, 'aimlab': 17899, 'ruwv': 28090, 'khubaib': 134053}
for k in ORDER:
    flag = 'OK' if len(raw[k]) == EXPECTED_RAW[k] else f'!! expected {EXPECTED_RAW[k]:,}'
    print(f'{k:8s} {len(raw[k]):>8,} rows  {flag}')

uci      <- Roman Urdu DataSet.csv  (20,229 rows)
ruwv     <- Roman Urdu reviews Dataset with English translation.csv  (28,090 rows)
skipped (not one of the 4 datasets): Roman Urdu words and English Translation with spelling variation.csv
skipped (not one of the 4 datasets): daraz-code-mixed-product-reviews.csv
aimlab   <- Hugging Face Aimlab/Sentiment-Analysis-Roman-Urdu  (17,899 rows)
khubaib  <- Hugging Face Khubaib01/RomanUrdu-NLP-Sentiment-Corpus  (134,053 rows)
uci        20,229 rows  OK
aimlab     17,899 rows  OK
ruwv       28,090 rows  OK
khubaib   134,053 rows  OK


## 4. Clean each corpus
- Labels mapped to `neg / neu / pos` (very positive → pos, very negative → neg, typo "Neative" → neg; anything else dropped).
- `norm` is used **only for matching duplicates**; models train on the original `text` (with phone numbers/emails masked).
- Sentences that appear with conflicting labels inside a corpus are dropped; other duplicates keep one copy.

In [ ]:
LABEL_MAP = {'positive': 'pos', 'pos': 'pos', 'very positive': 'pos',
             'negative': 'neg', 'neg': 'neg', 'very negative': 'neg', 'neative': 'neg',
             'neutral': 'neu'}
PHONE = re.compile(r'(?:\+?92|0)3\d{2}[\s-]?\d{7}|\b\d{10,13}\b')
EMAIL = re.compile(r'[\w.+-]+@[\w-]+\.[\w.]+')

def norm(t):
    t = unicodedata.normalize('NFKC', str(t)).lower()
    t = re.sub(r'https?://\S+|www\.\S+|@\w+', ' ', t)
    t = re.sub(r'(.)\1{2,}', r'\1\1', t)
    t = re.sub(r'[^\w\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def clean(df, name):
    rep = {'rows': len(df)}
    df = df.copy()
    rep['missing_text'] = int(df['text'].isna().sum())   # empty cells in the file
    df = df.dropna(subset=['text'])
    df['label'] = df['raw_label'].astype(str).str.strip().str.lower().map(LABEL_MAP)
    rep['bad_label'] = int(df['label'].isna().sum())
    df = df.dropna(subset=['label'])
    df['text'] = df['text'].astype(str)
    df['norm'] = df['text'].map(norm)
    rep['empty'] = int((df['norm'] == '').sum())
    df = df[df['norm'] != '']
    n_lab = df.groupby('norm')['label'].nunique()
    conflict = n_lab[n_lab > 1].index
    rep['conflict_rows'] = int(df['norm'].isin(conflict).sum())
    df = df[~df['norm'].isin(conflict)]
    rep['dup_rows'] = int(df['norm'].duplicated().sum())
    df = df.drop_duplicates('norm').reset_index(drop=True)
    rep['pii_rows'] = int(df['text'].str.contains(PHONE).sum() + df['text'].str.contains(EMAIL).sum())
    df['text'] = df['text'].str.replace(EMAIL, '<EMAIL>', regex=True).str.replace(PHONE, '<PHONE>', regex=True)
    df['src'] = name
    rep['after_clean'] = len(df)
    return df[['src', 'text', 'norm', 'label']], rep

cleaned, reports = {}, {}
for name in ORDER:
    cleaned[name], reports[name] = clean(raw[name], name)
pd.DataFrame(reports).T

,rows,missing_text,bad_label,empty,conflict_rows,dup_rows,pii_rows,after_clean
uci,20229,1,0,165,187,500,14,19376
aimlab,17899,0,1,4,52,41,3,17801
ruwv,28090,0,0,0,84,4660,23,23346
khubaib,134053,1,0,285,10695,24361,40,98711


## 5. Save shared sentences (for RQ3), then remove them from later corpora
Exact overlaps are saved with both corpora's labels. Then each corpus loses every sentence that already appeared
(exactly or as a near-duplicate) in a corpus earlier in the priority order.

In [ ]:
order = ORDER
rows = []
for a, b in itertools.combinations(order, 2):
    m = cleaned[a][['norm', 'text', 'label']].merge(cleaned[b][['norm', 'label']], on='norm', suffixes=('_a', '_b'))
    m.insert(0, 'corpus_a', a); m.insert(1, 'corpus_b', b)
    rows.append(m)
overlaps = pd.concat(rows, ignore_index=True)
overlaps.to_csv(f'{OUT}/overlaps/exact_overlaps.csv', index=False)
agree = (overlaps.assign(agree=overlaps.label_a == overlaps.label_b)
         .groupby(['corpus_a', 'corpus_b'])['agree'].agg(shared='size', agreement='mean').round(3))
print(agree)

t0 = time.time()
allc = pd.concat([cleaned[k] for k in order], ignore_index=True)
longc = allc[allc['norm'].str.split().str.len() >= NEAR_DUP_MIN_WORDS].reset_index(drop=True)
X = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True,
                    dtype=np.float32).fit_transform(longc['norm'])
dist, idx = NearestNeighbors(n_neighbors=8, metric='cosine', n_jobs=-1).fit(X).kneighbors(X)
rank = {k: i for i, k in enumerate(order)}
src_arr, norm_arr = longc['src'].values, longc['norm'].values
near_earlier, near_pairs = set(), []
for i in range(len(longc)):
    for d, j in zip(dist[i][1:], idx[i][1:]):
        if 1 - d >= NEAR_DUP_COS and src_arr[j] != src_arr[i]:
            near_pairs.append((src_arr[i], norm_arr[i], src_arr[j], norm_arr[j], round(1 - d, 3)))
            if rank[src_arr[j]] < rank[src_arr[i]]:
                near_earlier.add((src_arr[i], norm_arr[i]))
pd.DataFrame(near_pairs, columns=['corpus_a', 'norm_a', 'corpus_b', 'norm_b', 'cosine']).to_csv(
    f'{OUT}/overlaps/near_duplicate_pairs.csv', index=False)
print(f'near-duplicate search took {time.time() - t0:.0f}s')

final, seen, summary = {}, set(), []
for k in order:
    df = cleaned[k]
    exact = df['norm'].isin(seen)
    near = np.array([(k, n) in near_earlier for n in df['norm']]) & ~exact.values
    final[k] = df[~exact.values & ~near].reset_index(drop=True)
    seen |= set(df['norm'])
    counts = final[k]['label'].value_counts().reindex(['neg', 'neu', 'pos'], fill_value=0)
    summary.append({'corpus': k, 'after_clean': len(df), 'removed_exact': int(exact.sum()),
                    'removed_near': int(near.sum()), 'final': len(final[k]), **counts.to_dict(),
                    'expected': EXPECTED_FINAL[k], 'matches_audit': len(final[k]) == EXPECTED_FINAL[k]})
summary = pd.DataFrame(summary)
summary.to_csv(f'{OUT}/results/corpus_summary.csv', index=False)
print(summary.to_string(index=False))
if not summary['matches_audit'].all():
    print('\n!! Final sizes differ from the audit. Check whether a source file changed before using these splits.')

                   shared  agreement
corpus_a corpus_b                   
aimlab   khubaib    15884      0.728
         ruwv          27      0.519
ruwv     khubaib      788      0.905
uci      aimlab      5245      1.000
         khubaib    15433      0.837
         ruwv         861      0.976
near-duplicate search took 1410s
 corpus  after_clean  removed_exact  removed_near  final   neg   neu   pos  expected  matches_audit
    uci        19376              0             0  19376  5231  8352  5793     19376           True
 aimlab        17801           5245          1203  11353  5268   458  5627     11353           True
   ruwv        23346            866          2080  20400  6917  6035  7448     20401          False
khubaib        98711          28006           124  70581 30410 26178 13993     70581           True

!! Final sizes differ from the audit. Check whether a source file changed before using these splits.


## 6. Fixed splits (80 / 10 / 10, stratified)
Saved once to Drive. **All later notebooks load these files — never re-split.**

In [ ]:
splits = {}
for k, df in final.items():
    tr, rest = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=SPLIT_SEED)
    va, te = train_test_split(rest, test_size=0.5, stratify=rest['label'], random_state=SPLIT_SEED)
    splits[k] = {'train': tr.reset_index(drop=True), 'val': va.reset_index(drop=True), 'test': te.reset_index(drop=True)}
    for s, d in splits[k].items():
        d.to_csv(f'{OUT}/splits/{k}_{s}.csv', index=False)
pd.DataFrame({k: {s: len(d) for s, d in v.items()} for k, v in splits.items()}).T

,train,val,test
uci,15500,1938,1938
aimlab,9082,1135,1136
ruwv,16320,2040,2040
khubaib,56464,7058,7059


## 7. Classical baselines
For each track, every source trains on the **same number of sentences** (the smallest training split in that track),
drawn by stratified sampling with each seed. The regularisation strength `C` is chosen on the **source** validation set only.
Each model is then tested on every corpus's test set in the track.

In [ ]:
MODELS = {
    'tfidf_word_lr': lambda: TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    'tfidf_char_lr': lambda: TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True,
                                             max_features=300_000),
}
C_GRID = [0.25, 1.0, 4.0]

def subset(df, classes):
    return df[df['label'].isin(classes)].reset_index(drop=True)

def sample_to(df, n, seed):
    if len(df) <= n:
        return df
    return train_test_split(df, train_size=n, stratify=df['label'], random_state=seed)[0]

results, preds = [], []
for track, (corpora, classes) in TRACKS.items():
    data = {k: {s: subset(splits[k][s], classes) for s in ['train', 'val', 'test']} for k in corpora}
    cap = min(len(data[k]['train']) for k in corpora)
    print(f'\n[{track}] classes={classes} training size per source = {cap:,}')
    for model_name, make_vec in MODELS.items():
        for src in corpora:
            for seed in SEEDS:
                tr = sample_to(data[src]['train'], cap, seed)
                vec = make_vec().fit(tr['text'])
                Xtr, Xva = vec.transform(tr['text']), vec.transform(data[src]['val']['text'])
                best = None
                for C in C_GRID:
                    clf = LogisticRegression(C=C, max_iter=3000, class_weight='balanced', random_state=seed)
                    clf.fit(Xtr, tr['label'])
                    f = f1_score(data[src]['val']['label'], clf.predict(Xva), average='macro', labels=classes)
                    if best is None or f > best[0]:
                        best = (f, C, clf)
                _, C, clf = best
                for tgt in corpora:
                    te = data[tgt]['test']
                    proba = clf.predict_proba(vec.transform(te['text']))
                    pred = clf.classes_[proba.argmax(1)]
                    per_class = f1_score(te['label'], pred, average=None, labels=classes)
                    results.append({'track': track, 'model': model_name, 'source': src, 'target': tgt, 'seed': seed,
                                    'C': C, 'n_train': len(tr), 'n_test': len(te),
                                    'macro_f1': f1_score(te['label'], pred, average='macro', labels=classes),
                                    'accuracy': accuracy_score(te['label'], pred),
                                    **{f'f1_{c}': v for c, v in zip(classes, per_class)}})
                    p = te[['text', 'label']].copy()
                    p['row'] = np.arange(len(te)); p['pred'] = pred
                    for j, c in enumerate(clf.classes_):
                        p[f'p_{c}'] = proba[:, j]
                    p['track'], p['model'], p['source'], p['target'], p['seed'] = track, model_name, src, tgt, seed
                    preds.append(p)
            print(f'  {model_name:14s} source={src:8s} done')

results = pd.DataFrame(results)
results.to_csv(f'{OUT}/results/baseline_results.csv', index=False)
preds = pd.concat(preds, ignore_index=True)
preds.to_parquet(f'{OUT}/predictions/baseline_predictions.parquet', index=False)
results.groupby(['track', 'model', 'source', 'target'])['macro_f1'].agg(['mean', 'std']).round(3)


[3class] classes=['neg', 'neu', 'pos'] training size per source = 15,500
  tfidf_word_lr  source=uci      done
  tfidf_word_lr  source=ruwv     done
  tfidf_word_lr  source=khubaib  done
  tfidf_char_lr  source=uci      done
  tfidf_char_lr  source=ruwv     done
  tfidf_char_lr  source=khubaib  done

[binary] classes=['neg', 'pos'] training size per source = 8,716
  tfidf_word_lr  source=uci      done
  tfidf_word_lr  source=aimlab   done
  tfidf_word_lr  source=ruwv     done
  tfidf_word_lr  source=khubaib  done
  tfidf_char_lr  source=uci      done
  tfidf_char_lr  source=aimlab   done
  tfidf_char_lr  source=ruwv     done
  tfidf_char_lr  source=khubaib  done


mean    std
track  model         source  target               
3class tfidf_char_lr khubaib khubaib  0.726  0.002
                             ruwv     0.489  0.009
                             uci      0.517  0.002
                     ruwv    khubaib  0.547  0.005
                             ruwv     0.663  0.003
                             uci      0.576  0.003
                     uci     khubaib  0.576  0.000
                             ruwv     0.560  0.000
                             uci      0.660  0.000
       tfidf_word_lr khubaib khubaib  0.711  0.005
                             ruwv     0.489  0.004
                             uci      0.508  0.001
                     ruwv    khubaib  0.499  0.005
                             ruwv     0.658  0.004
                             uci      0.550  0.004
                     uci     khubaib  0.523  0.000
                             ruwv     0.546  0.000
                             uci      0.642  0.000
binary tfidf_char_lr aimlab  aimlab   0.827  0.000
                             khubaib  0.765  0.000
                             ruwv     0.695  0.000
                             uci      0.699  0.000
                     khubaib aimlab   0.717  0.009
                             khubaib  0.847  0.004
                             ruwv     0.653  0.007
                             uci      0.645  0.007
                     ruwv    aimlab   0.710  0.018
                             khubaib  0.705  0.006
                             ruwv     0.778  0.008
                             uci      0.777  0.011
                     uci     aimlab   0.754  0.003
                             khubaib  0.751  0.000
                             ruwv     0.785  0.001
                             uci      0.799  0.006
       tfidf_word_lr aimlab  aimlab   0.816  0.000
                             khubaib  0.741  0.000
                             ruwv     0.685  0.000
                             uci      0.658  0.000
                     khubaib aimlab   0.708  0.004
                             khubaib  0.836  0.001
                             ruwv     0.670  0.009
                             uci      0.653  0.009
                     ruwv    aimlab   0.693  0.006
                             khubaib  0.683  0.012
                             ruwv     0.778  0.003
                             uci      0.765  0.006
                     uci     aimlab   0.723  0.002
                             khubaib  0.730  0.004
                             ruwv     0.763  0.005
                             uci      0.789  0.011

## 8. Generalisation drop with 95% bootstrap confidence intervals
drop(S→T) = macro-F1 on S's own test set − macro-F1 on T's test set, averaged over seeds.
The CI resamples both test sets 1,000 times. A CI that excludes 0 means the drop is real, not noise.

In [ ]:
def macro_f1_fast(y, p, k):
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    denom = 2 * tp + fp + fn
    return np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0).mean()

rng = np.random.default_rng(0)
drops = []
for (track, model, src), g in preds.groupby(['track', 'model', 'source']):
    classes = TRACKS[track][1]; k = len(classes); cid = {c: i for i, c in enumerate(classes)}
    def arrays(tgt):
        sub = g[g['target'] == tgt].sort_values(['seed', 'row'])
        y = sub[sub['seed'] == SEEDS[0]]['label'].map(cid).values
        P = np.stack([sub[sub['seed'] == s]['pred'].map(cid).values for s in SEEDS])
        return y, P
    yS, PS = arrays(src)
    for tgt in TRACKS[track][0]:
        if tgt == src:
            continue
        yT, PT = arrays(tgt)
        point = np.mean([macro_f1_fast(yS, p, k) for p in PS]) - np.mean([macro_f1_fast(yT, p, k) for p in PT])
        boot = []
        for _ in range(BOOTSTRAP):
            iS = rng.integers(0, len(yS), len(yS)); iT = rng.integers(0, len(yT), len(yT))
            boot.append(np.mean([macro_f1_fast(yS[iS], p[iS], k) for p in PS])
                        - np.mean([macro_f1_fast(yT[iT], p[iT], k) for p in PT]))
        lo, hi = np.percentile(boot, [2.5, 97.5])
        in_f1 = np.mean([macro_f1_fast(yS, p, k) for p in PS])
        drops.append({'track': track, 'model': model, 'source': src, 'target': tgt,
                      'in_corpus_f1': round(in_f1, 4), 'cross_f1': round(in_f1 - point, 4),
                      'drop': round(point, 4), 'drop_ci_low': round(lo, 4), 'drop_ci_high': round(hi, 4),
                      'relative_drop_%': round(100 * point / in_f1, 1)})
drops = pd.DataFrame(drops)
drops.to_csv(f'{OUT}/results/baseline_drops.csv', index=False)
drops

,track,model,source,target,in_corpus_f1,cross_f1,drop,drop_ci_low,drop_ci_high,relative_drop_%
0,3class,tfidf_char_lr,khubaib,uci,0.7257,0.5170,0.2088,0.1864,0.2303,28.8
1,3class,tfidf_char_lr,khubaib,ruwv,0.7257,0.4895,0.2363,0.2129,0.2573,32.6
2,3class,tfidf_char_lr,ruwv,uci,0.6627,0.5755,0.0871,0.0588,0.1155,13.1
3,3class,tfidf_char_lr,ruwv,khubaib,0.6627,0.5472,0.1155,0.0936,0.1394,17.4
4,3class,tfidf_char_lr,uci,ruwv,0.6596,0.5599,0.0997,0.0701,0.1303,15.1
5,3class,tfidf_char_lr,uci,khubaib,0.6596,0.5764,0.0832,0.0603,0.1085,12.6
6,3class,tfidf_word_lr,khubaib,uci,0.7107,0.5081,0.2026,0.1799,0.2244,28.5
7,3class,tfidf_word_lr,khubaib,ruwv,0.7107,0.4890,0.2218,0.2006,0.2440,31.2
8,3class,tfidf_word_lr,ruwv,uci,0.6583,0.5505,0.1079,0.0771,0.1382,16.4
9,3class,tfidf_word_lr,ruwv,khubaib,0.6583,0.4986,0.1598,0.1375,0.1824,24.3


## 9. Done — what was saved
- `splits/` — fixed train/val/test CSVs for each corpus (used by every later notebook)
- `overlaps/` — exact shared sentences with both labels, and near-duplicate pairs (RQ3)
- `results/corpus_summary.csv` — sizes and class counts; check `matches_audit` is True for all four
- `results/baseline_results.csv` — every model × source × target × seed score
- `results/baseline_drops.csv` — the generalisation-drop table with confidence intervals
- `predictions/baseline_predictions.parquet` — every prediction, for the attribution and error analyses

**Send me** `corpus_summary.csv` and `baseline_drops.csv` after the run.

## 10. Download everything (keep this zip — notebook 2 needs the `splits/` folder)

In [ ]:
import shutil
shutil.make_archive('/content/roman_urdu_paper_outputs', 'zip', OUT)
try:
    from google.colab import files
    files.download('/content/roman_urdu_paper_outputs.zip')
except ImportError:
    print('saved roman_urdu_paper_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Part 2 (mBERT and XLM-R)

In [ ]:
import os, glob
for f in glob.glob('/content/*.zip'): print(os.path.basename(f), os.path.getsize(f))

In [ ]:
from google.colab import drive
import zipfile
import os

# 1. Mount your Google Drive (a pop-up will ask you to sign in and grant permission)
drive.mount('/content/drive')

# 2. Define paths (Change 'datasets.zip' if your file has a different name)
zip_path = '/content/roman_urdu_paper_outputs (1).zip'
extract_path = '/content/datasets'

# 3. Unzip the file directly into Colab's fast local storage
print("Extracting files...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Done! Your files are ready at:", extract_path)

Mounted at /content/drive
Extracting files...
Done! Your files are ready at: /content/datasets


In [ ]:
import os
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

OUT = '/content/datasets'
SEEDS = [13, 42, 2026]                  # same seeds as the baselines
TRACKS = {'3class': (['uci', 'ruwv', 'khubaib'], ['neg', 'neu', 'pos']),
          'binary': (['uci', 'aimlab', 'ruwv', 'khubaib'], ['neg', 'pos'])}
MODELS_HF = {'xlmr': 'xlm-roberta-base', 'mbert': 'bert-base-multilingual-cased'}
MAX_LEN, BATCH, LR, EPOCHS, WARMUP = 128, 32, 2e-5, 3, 0.1

PRED_DIR = f'{OUT}/predictions/transformers'
RESULTS_CSV = f'{OUT}/results/transformer_results.csv'
os.makedirs(PRED_DIR, exist_ok=True); os.makedirs(f'{OUT}/results', exist_ok=True)

# keep_default_na=False: otherwise sentences like "NA" or "null" would be read as empty
splits = {k: {s: pd.read_csv(f'{OUT}/splits/{k}_{s}.csv', keep_default_na=False) for s in ['train', 'val', 'test']}
          for k in ['uci', 'aimlab', 'ruwv', 'khubaib']}
print(pd.DataFrame({k: {s: len(d) for s, d in v.items()} for k, v in splits.items()}).T)

def subset(df, classes):
    return df[df['label'].isin(classes)].reset_index(drop=True)

def sample_to(df, n, seed):            # identical to Part 1, so transformers see the same training sentences
    if len(df) <= n:
        return df
    return train_test_split(df, train_size=n, stratify=df['label'], random_state=seed)[0]

         train   val  test
uci      15500  1938  1938
aimlab    9082  1135  1136
ruwv     16320  2040  2040
khubaib  56464  7058  7059


In [ ]:
import torch
import random
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def batches(texts, labels, tok, bs, shuffle=False, seed=0):
    idx = np.arange(len(texts))
    if shuffle:
        np.random.default_rng(seed).shuffle(idx)
    for i in range(0, len(idx), bs):
        b = idx[i:i + bs]
        enc = tok([texts[j] for j in b], truncation=True, max_length=MAX_LEN, padding=True, return_tensors='pt')
        enc = {k: v.cuda() for k, v in enc.items()}
        yield enc, (torch.tensor(labels[b], dtype=torch.long).cuda() if labels is not None else None)

@torch.no_grad()
def predict(model, tok, texts):
    model.eval()
    order = np.argsort([len(t) for t in texts])          # similar lengths together = faster
    probs = []
    for enc, _ in batches([texts[i] for i in order], None, tok, 128):
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(**enc).logits
        probs.append(torch.softmax(logits.float(), -1).cpu().numpy())
    probs = np.concatenate(probs)
    out = np.empty_like(probs); out[order] = probs
    return out

def run_one(track, mkey, src, seed):
    corpora, classes = TRACKS[track]
    cid = {c: i for i, c in enumerate(classes)}
    data = {k: {s: subset(splits[k][s], classes) for s in ['train', 'val', 'test']} for k in corpora}
    cap = min(len(data[k]['train']) for k in corpora)
    tr = sample_to(data[src]['train'], cap, seed)
    X, y = tr['text'].tolist(), tr['label'].map(cid).values

    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODELS_HF[mkey])
    model = AutoModelForSequenceClassification.from_pretrained(MODELS_HF[mkey], num_labels=len(classes)).cuda()
    weights = len(y) / (len(classes) * np.bincount(y, minlength=len(classes)))
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).cuda())
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    # Calculate steps based on target batch size of 32
    target_batch = 32
    steps = EPOCHS * int(np.ceil(len(X) / target_batch))
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler('cuda')

    # Gradient Accumulation parameters
    accum_steps = int(target_batch / BATCH)

    val = data[src]['val']
    best_f1, best_ep, best_probs = -1, None, None
    for ep in range(1, EPOCHS + 1):
        model.train()
        opt.zero_grad()
        for step_i, (enc, yb) in enumerate(batches(X, y, tok, BATCH, shuffle=True, seed=seed * 100 + ep), 1):
            with torch.autocast('cuda', dtype=torch.float16):
                logits = model(**enc).logits
            loss = loss_fn(logits.float(), yb) / accum_steps

            scaler.scale(loss).backward()

            if step_i % accum_steps == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad()

        pv = predict(model, tok, val['text'].tolist())
        f = f1_score(val['label'], np.array(classes)[pv.argmax(1)], average='macro', labels=classes)
        print(f'    epoch {ep}: source val macro-F1 = {f:.4f}')
        if f > best_f1:
            best_f1, best_ep = f, ep
            best_probs = {t: predict(model, tok, data[t]['test']['text'].tolist()) for t in corpora}

    rows, preds = [], []
    for tgt in corpora:
        te, P = data[tgt]['test'], best_probs[tgt]
        pred = np.array(classes)[P.argmax(1)]
        per_class = f1_score(te['label'], pred, average=None, labels=classes)
        rows.append({'track': track, 'model': mkey, 'source': src, 'target': tgt, 'seed': seed,
                     'best_epoch': best_ep, 'val_f1': round(best_f1, 4), 'n_train': len(tr), 'n_test': len(te),
                     'macro_f1': f1_score(te['label'], pred, average='macro', labels=classes),
                     'accuracy': accuracy_score(te['label'], pred),
                     **{f'f1_{c}': v for c, v in zip(classes, per_class)}})
        p = te[['text', 'label']].copy()
        p['row'] = np.arange(len(te)); p['pred'] = pred
        for j, c in enumerate(classes):
            p[f'p_{c}'] = P[:, j]
        p['track'], p['model'], p['source'], p['target'], p['seed'] = track, mkey, src, tgt, seed
        preds.append(p)
    del model; torch.cuda.empty_cache()
    return pd.DataFrame(rows), pd.concat(preds, ignore_index=True)

In [ ]:
import os
import shutil
import time
import gc
import torch

# 1. Clear memory from the previous crashed run
gc.collect()
torch.cuda.empty_cache()

# 2. Set memory-efficient parameters for 5-epoch run
MAX_LEN = 96  # Reduced from 128 to dramatically save memory
BATCH = 8     # Reduced from 16 to guarantee we stay below the GPU memory ceiling
EPOCHS = 5
WARMUP = 0.1

PRED_DIR = f'{OUT}/predictions/transformers_5ep'
RESULTS_CSV = f'{OUT}/results/transformer_results_5ep.csv'
os.makedirs(PRED_DIR, exist_ok=True)

# 3. Running the plan
PLAN_5EP = [('3class', 'xlmr', src, 13) for src in TRACKS['3class'][0]]

for i, (track, mkey, src, seed) in enumerate(PLAN_5EP, 1):
    fname = f'{PRED_DIR}/{track}_{mkey}_{src}_{seed}.parquet'
    if os.path.exists(fname):
        continue

    print(f'[{i}/{len(PLAN_5EP)}] 5 epochs: {track} {mkey} source={src} seed={seed}')
    t0 = time.time()
    res, pr = run_one(track, mkey, src, seed)

    pr.to_parquet(fname, index=False)
    res.to_csv(RESULTS_CSV, mode='a', header=not os.path.exists(RESULTS_CSV), index=False)
    shutil.make_archive('/content/roman_urdu_paper_outputs', 'zip', OUT)

    print(f'    done in {(time.time() - t0) / 60:.1f} min | macro-F1 by target:',
          dict(zip(res['target'], res['macro_f1'].round(3))))

[1/3] 5 epochs: 3class xlmr source=uci seed=13


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

... (3 outputs left out)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

    epoch 1: source val macro-F1 = 0.5846
    epoch 2: source val macro-F1 = 0.6204
    epoch 3: source val macro-F1 = 0.6543
    epoch 4: source val macro-F1 = 0.6591
    epoch 5: source val macro-F1 = 0.6543
    done in 14.3 min | macro-F1 by target: {'uci': 0.533, 'ruwv': 0.656, 'khubaib': 0.543}
[3/3] 5 epochs: 3class xlmr source=khubaib seed=13


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

    epoch 1: source val macro-F1 = 0.6923
    epoch 2: source val macro-F1 = 0.6761
    epoch 3: source val macro-F1 = 0.7251
    epoch 4: source val macro-F1 = 0.7354
    epoch 5: source val macro-F1 = 0.7346
    done in 9.7 min | macro-F1 by target: {'uci': 0.525, 'ruwv': 0.431, 'khubaib': 0.744}


In [ ]:
import pandas as pd
OUT = '/content/datasets'
r3 = pd.read_csv(f'{OUT}/results/transformer_results.csv')
r5 = pd.read_csv(f'{OUT}/results/transformer_results_5ep.csv')
key = ['track', 'model', 'source', 'target', 'seed']
cmp = r3.merge(r5, on=key, suffixes=('_3ep', '_5ep'))
cmp[key + ['best_epoch_5ep', 'macro_f1_3ep', 'macro_f1_5ep']]

,track,model,source,target,seed,best_epoch_5ep,macro_f1_3ep,macro_f1_5ep
0,3class,xlmr,uci,uci,13,5,0.636871,0.664222
1,3class,xlmr,uci,ruwv,13,5,0.527514,0.542528
2,3class,xlmr,uci,khubaib,13,5,0.624070,0.621357


In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/roman_urdu_paper_outputs', 'zip', OUT)
files.download('/content/roman_urdu_paper_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Order: XLM-R 3-class first (most important), then XLM-R binary, mBERT 3-class, mBERT binary.
# Each finished run is saved immediately; re-running this cell skips runs that are already done.
PLAN = [(track, m, src, seed) for m in ['xlmr', 'mbert'] for track in ['3class', 'binary']
        for src in TRACKS[track][0] for seed in SEEDS]
print(f'{len(PLAN)} runs planned')
for i, (track, mkey, src, seed) in enumerate(PLAN, 1):
    fname = f'{PRED_DIR}/{track}_{mkey}_{src}_{seed}.parquet'
    if os.path.exists(fname):
        continue
    print(f'[{i}/{len(PLAN)}] {track} {mkey} source={src} seed={seed}')
    t0 = time.time()
    res, pr = run_one(track, mkey, src, seed)
    pr.to_parquet(fname, index=False)
    res.to_csv(RESULTS_CSV, mode='a', header=not os.path.exists(RESULTS_CSV), index=False)
    print(f'    done in {(time.time() - t0) / 60:.1f} min | macro-F1 by target:',
          dict(zip(res['target'], res['macro_f1'].round(3))))

42 runs planned
[1/42] 3class xlmr source=uci seed=13


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

... (92 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    epoch 1: source val macro-F1 = 0.7909
    epoch 2: source val macro-F1 = 0.8056
    epoch 3: source val macro-F1 = 0.8345
    done in 2.9 min | macro-F1 by target: {'uci': 0.652, 'aimlab': 0.662, 'ruwv': 0.636, 'khubaib': 0.819}
[42/42] binary mbert source=khubaib seed=2026


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    epoch 1: source val macro-F1 = 0.7829
    epoch 2: source val macro-F1 = 0.7931
    epoch 3: source val macro-F1 = 0.8385
    done in 2.9 min | macro-F1 by target: {'uci': 0.626, 'aimlab': 0.673, 'ruwv': 0.608, 'khubaib': 0.826}


In [ ]:
import shutil
shutil.make_archive('/content/roman_urdu_paper_outputs', 'zip', OUT)
from google.colab import files
files.download('/content/roman_urdu_paper_outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
def macro_f1_fast(y, p, k):
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    denom = 2 * tp + fp + fn
    return np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0).mean()

files_ = glob.glob(f'{PRED_DIR}/*.parquet')
tp = pd.concat([pd.read_parquet(f) for f in files_], ignore_index=True)
rng = np.random.default_rng(0)
drops = []
for (track, model, src), g in tp.groupby(['track', 'model', 'source']):
    if g['seed'].nunique() < len(SEEDS):
        print(f'skipping {track} {model} {src}: only {g["seed"].nunique()} of {len(SEEDS)} seeds done')
        continue
    classes = TRACKS[track][1]; k = len(classes); cid = {c: i for i, c in enumerate(classes)}
    def arrays(tgt):
        sub = g[g['target'] == tgt].sort_values(['seed', 'row'])
        y = sub[sub['seed'] == SEEDS[0]]['label'].map(cid).values
        P = np.stack([sub[sub['seed'] == s]['pred'].map(cid).values for s in SEEDS])
        return y, P
    yS, PS = arrays(src)
    in_f1 = np.mean([macro_f1_fast(yS, p, k) for p in PS])
    for tgt in TRACKS[track][0]:
        if tgt == src:
            continue
        yT, PT = arrays(tgt)
        cross = np.mean([macro_f1_fast(yT, p, k) for p in PT])
        boot = []
        for _ in range(1000):
            iS = rng.integers(0, len(yS), len(yS)); iT = rng.integers(0, len(yT), len(yT))
            boot.append(np.mean([macro_f1_fast(yS[iS], p[iS], k) for p in PS])
                        - np.mean([macro_f1_fast(yT[iT], p[iT], k) for p in PT]))
        lo, hi = np.percentile(boot, [2.5, 97.5])
        drops.append({'track': track, 'model': model, 'source': src, 'target': tgt,
                      'in_corpus_f1': round(in_f1, 4), 'cross_f1': round(cross, 4), 'drop': round(in_f1 - cross, 4),
                      'drop_ci_low': round(lo, 4), 'drop_ci_high': round(hi, 4),
                      'relative_drop_%': round(100 * (in_f1 - cross) / in_f1, 1)})
drops = pd.DataFrame(drops)
drops.to_csv(f'{OUT}/results/transformer_drops.csv', index=False)
drops

,track,model,source,target,in_corpus_f1,cross_f1,drop,drop_ci_low,drop_ci_high,relative_drop_%
0,3class,mbert,khubaib,uci,0.7154,0.4877,0.2278,0.2032,0.2498,31.8
1,3class,mbert,khubaib,ruwv,0.7154,0.4428,0.2727,0.2503,0.2942,38.1
2,3class,mbert,ruwv,uci,0.6550,0.5090,0.1460,0.1202,0.1741,22.3
3,3class,mbert,ruwv,khubaib,0.6550,0.5136,0.1414,0.1215,0.1637,21.6
4,3class,mbert,uci,ruwv,0.6287,0.5253,0.1034,0.0755,0.1310,16.5
5,3class,mbert,uci,khubaib,0.6287,0.5953,0.0334,0.0117,0.0559,5.3
6,3class,xlmr,khubaib,uci,0.7240,0.5062,0.2178,0.1937,0.2391,30.1
7,3class,xlmr,khubaib,ruwv,0.7240,0.4294,0.2947,0.2730,0.3170,40.7
8,3class,xlmr,ruwv,uci,0.6441,0.5429,0.1012,0.0756,0.1288,15.7
9,3class,xlmr,ruwv,khubaib,0.6441,0.5700,0.0741,0.0519,0.0965,11.5


In [ ]:
MAX_LEN, BATCH, LR, EPOCHS, WARMUP = 128, 32, 2e-5, 5, 0.1

In [ ]:
   import shutil
   shutil.rmtree(PRED_DIR); os.makedirs(PRED_DIR)
   if os.path.exists(RESULTS_CSV): os.remove(RESULTS_CSV)
   print('cleared')

cleared
